<a href="https://colab.research.google.com/github/isabelcordova-hash/Analitica-Empresarial-Integrada/blob/main/LAB_D11_AEI_IC%C3%93RDOVA_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO N.° 11 — Analítica Prescriptiva: Optimización

**Semana 11 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía  
**Dataset real:** Online Retail II — UCI Machine Learning Repository  
**Herramienta:** PuLP

### Pregunta de negocio
**¿Cómo debe distribuirse una capacidad limitada de inventario entre productos para maximizar el valor potencial de venta sin descuidar niveles mínimos de servicio?**

> El objetivo no es solo obtener `Optimal`. Debe demostrarse cuánto mejora la decisión frente a una regla intuitiva.

---

**Estudiante:** Córdova Agurto, Isabel
**Sección:** 6C28  
**Fecha:** 28/10/2026

---

- Repositorio:
- Uso de IA declarado: Se utilizó Qwen para ajustes de redacción.

In [1]:
# Instala en Colab las librerías que no vienen por defecto; la versión de PuLP queda fija para repetir resultados
%pip install -q "ucimlrepo>=0.0.7" "pulp==3.3.0" plotly openpyxl

# Librerías: pandas y numpy para los datos, PuLP para optimizar y plotly para graficar
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import pulp as pl
from ucimlrepo import fetch_ucirepo
# Se registran las versiones para que otra persona pueda repetir el resultado
print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· PuLP:", pl.__version__)
# Permite ver hasta 50 columnas al imprimir tablas
pd.set_option("display.max_columns", 50)

Python: 3.13.18 · pandas: 2.2.3 · PuLP: 3.3.0


**Interpretación de resultados**

La salida corresponde solo a la revisión del entorno de ejecución: se listan las versiones de las herramientas empleadas (Python 3.13.18, pandas 2.2.3 y PuLP 3.3.0). Todavía no hay resultados analíticos ni indicadores de negocio, pero dejar registradas estas versiones es una buena práctica de trazabilidad: permite que cualquier persona repita el laboratorio en las mismas condiciones y obtenga resultados idénticos, sin sorpresas por cambios en las librerías.

# 1. Descarga de datos reales y trazabilidad

In [2]:
# API UCI; si falla, usar el ZIP oficial. La carga manual queda como último recurso.
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
from pathlib import Path

try:
    retail = fetch_ucirepo(id=502)
    df = retail.data.original.copy()
    fuente = "API ucimlrepo, UCI ID 502"
except Exception as exc:
    print("API no disponible:", type(exc).__name__)
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as response:
            with ZipFile(BytesIO(response.read())) as z:
                name = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
                df = pd.concat(pd.read_excel(BytesIO(z.read(name)), sheet_name=None).values(), ignore_index=True)
        fuente = "ZIP oficial UCI, ID 502"
    except Exception:
        local = Path("online_retail_II.xlsx")
        if not local.exists():
            raise FileNotFoundError("Descargue online_retail_II.xlsx desde UCI y súbalo a Colab.")
        df = pd.concat(pd.read_excel(local, sheet_name=None).values(), ignore_index=True)
        fuente = "Archivo local descargado de UCI"
print("Fuente:", fuente, "· ejecución UTC:", pd.Timestamp.now(tz="UTC").isoformat())
print("Dimensión:", df.shape, "· columnas:", df.columns.tolist())
display(df.head())

API no disponible: DatasetNotFoundError
Fuente: ZIP oficial UCI, ID 502 · ejecución UTC: 2026-11-01T21:51:42.875274+00:00
Dimensión: (1067371, 8) · columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Interpretación de resultados**

La salida indica que el dataset "Online Retail II" se obtuvo mediante el plan de contingencia, es decir, la descarga del ZIP oficial de UCI, porque la conexión por API falló al inicio. Se cargaron 1.067.371 registros distribuidos en 8 columnas, entre ellas StockCode, Quantity, Price e InvoiceDate, que son centrales para armar el modelo. Las primeras filas confirman la estructura esperada y el origen de los datos (transacciones del Reino Unido de diciembre de 2009), de modo que se puede avanzar con la normalización de nombres y el control de calidad.

## Normalización de nombres

In [3]:
# El archivo trae nombres de columna distintos según la fuente (Invoice o InvoiceNo, Price o UnitPrice); esta función busca cuál se usa
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_",""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_","")
        if key in normalized:
            return normalized[key]
    return None

# Se traduce cada columna a un nombre único que usa el resto del notebook
mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "Country": find_col(df.columns, ["Country"]),
}

# Si falta una columna indispensable, se detiene aquí en lugar de fallar más adelante
required = ["InvoiceNo","StockCode","Quantity","InvoiceDate","UnitPrice","Country"]
missing = [k for k in required if mapping[k] is None]
if missing:
    raise ValueError(f"No se identificaron columnas obligatorias: {missing}")

# Se renombran las columnas y se muestra la correspondencia encontrada
df = df.rename(columns={v:k for k,v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'Country': 'Country'}


**Interpretación de resultados**

El resultado es el diccionario que relaciona cada nombre estándar del notebook (como 'InvoiceNo' o 'UnitPrice') con la columna que realmente trae el archivo (como 'Invoice' o 'Price'). Muestra que el código reconoció correctamente las variantes de nomenclatura y las unificó. Esta estandarización mantiene coherentes las fases siguientes de limpieza, filtrado y agregación, y evita errores de referencia si el archivo fuente de UCI llega con otros nombres de columna.

# 2. Control de calidad

In [4]:
# Se convierten fecha, cantidad y precio a su tipo correcto; lo que no se pueda convertir queda vacío
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
# Se quitan los registros sin código de producto y se limpian espacios
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
# Las facturas que empiezan con C son cancelaciones: restan demanda y no deben contarse como ventas
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
# Regla de limpieza: solo ventas válidas (cantidad y precio positivos, con fecha, producto y país)
clean = df.loc[(~cancel) & df["InvoiceDate"].notna() &
               df["StockCode"].ne("") & df["Country"].notna() &
               (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
# Se conserva el Reino Unido, el mercado principal del archivo
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
# Se registra cuántas cancelaciones había y cuántas filas válidas quedaron
print("Cancelaciones:", int(cancel.sum()), "· filas válidas UK:", len(uk))
print("Periodo:", uk["InvoiceDate"].min(), "→", uk["InvoiceDate"].max())

Cancelaciones: 19494 · filas válidas UK: 958501
Periodo: 2009-12-01 07:45:00 → 2011-12-09 12:49:00


**Interpretación de resultados**

La salida resume la limpieza: se excluyeron 19.494 transacciones de cancelación (facturas que empiezan con "C"), que son devoluciones o anulaciones y que, de sumarse como ventas, deformarían la demanda real. Luego de aplicar las reglas de calidad (cantidades y precios positivos, fechas válidas) y quedarse solo con el Reino Unido, resultan 958.501 filas válidas. El periodo cubierto va de diciembre de 2009 a diciembre de 2011, es decir, dos años de historia: una base depurada y sólida para calcular la demanda semanal y los parámetros del modelo de optimización.

### Pregunta 1
¿Por qué es incorrecto usar directamente la suma de Quantity sin revisar cancelaciones y precios?

**Respuesta:**

Sumar Quantity tal cual, sin examinar cancelaciones ni precios, es un error porque se contarían movimientos que no son ventas reales. El código detecta 19,494 cancelaciones (facturas que empiezan con "C"), que son devoluciones o anulaciones y restan demanda en vez de aportarla. Además, si no se descartan precios cero o negativos ni cantidades no positivas, entrarían valores atípicos, errores de registro o ajustes contables que distorsionarían la demanda estimada y el valor potencial de venta. El proceso de limpieza deja solo transacciones válidas, con fecha, producto, país, cantidad y precio positivos, de modo que los parámetros del modelo representen la demanda real del mercado.

# 3. Construcción de parámetros reales

In [5]:
# Últimas 12 semanas COMPLETAS: lunes a domingo.
# El archivo termina el 9-dic-2011 (viernes); se excluye esa semana parcial.
last_obs = uk["InvoiceDate"].max().normalize()
last_complete_sunday = last_obs - pd.Timedelta(days=last_obs.weekday() + 1)
first_monday = last_complete_sunday - pd.Timedelta(weeks=12) + pd.Timedelta(days=1)
recent = uk.loc[
    (uk["InvoiceDate"] >= first_monday) &
    (uk["InvoiceDate"] < last_complete_sunday + pd.Timedelta(days=1))
].copy()
recent["week"] = recent["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
assert recent["week"].nunique() == 12, "Revisar cobertura de las 12 semanas."

weekly = recent.groupby(["StockCode", "week"], as_index=False).agg(
    units=("Quantity", "sum"), price=("UnitPrice", "median")
)
sku_params = weekly.groupby("StockCode").agg(
    weeks_present=("week", "nunique"),
    demand_total=("units", "sum")
).reset_index()
sku_params["weekly_demand"] = sku_params["demand_total"] / 12
sku_params["price"] = sku_params["StockCode"].map(
    recent.groupby("StockCode")["UnitPrice"].median()
)
eligible = sku_params.loc[sku_params["weeks_present"] >= 8].copy()
if len(eligible) < 10:
    eligible = sku_params.loc[sku_params["weeks_present"] >= 6].copy()
if len(eligible) < 10:
    raise ValueError("Menos de 10 SKU elegibles; revisar datos y regla de selección.")
top = eligible.sort_values(["demand_total", "StockCode"], ascending=[False, True]).head(10).copy()
top["demanda"] = np.maximum(np.ceil(top["weekly_demand"]), 1).astype(int)
parametros = top.set_index("StockCode")[["demanda", "price", "weeks_present", "demand_total"]].copy()
print("Ventana:", first_monday.date(), "a", last_complete_sunday.date())
display(parametros)
print("Demanda esperada total:", int(parametros["demanda"].sum()))

Ventana: 2011-09-12 a 2011-12-04


,demanda,price,weeks_present,demand_total
StockCode,,,,
22197,1809,0.85,12,21703
84077,1223,0.29,12,14676
85099B,1175,2.08,12,14098
22086,1080,2.95,12,12951
23084,959,2.08,12,11505
84879,860,1.69,12,10310
22952,765,0.55,12,9177
22577,746,0.29,12,8948
22578,693,0.29,12,8305


Demanda esperada total: 9982


**Interpretación de resultados**

Se obtienen los parámetros reales, calculados con datos de UCI, de los 10 SKU elegidos sobre una ventana de 12 semanas completas (del 12 de septiembre al 4 de diciembre de 2011). La columna demanda es el promedio semanal por producto, redondeado hacia arriba para cubrir como mínimo lo observado. La columna price es la mediana de precios en esas 12 semanas, que resiste valores extremos y promociones puntuales. Las columnas weeks_present y demand_total indican que todos los SKU aparecieron las 12 semanas y cuántas unidades acumularon en ese lapso. La demanda esperada total, 9,982 unidades, es la suma de las demandas semanales de los 10 productos y sirve de base para fijar la capacidad (70 % de ese total) del modelo de optimización.

### Pregunta 2
¿Qué significa una semana sin ventas observadas para un SKU y por qué no prueba demanda cero?

**Respuesta:**

Que un SKU no registre ventas en una semana no demuestra que su demanda haya sido cero, porque los datos solo reflejan lo que efectivamente se vendió, no lo que los clientes querían comprar. Esa ausencia puede deberse a falta de stock (demanda censurada), problemas de abastecimiento o limitaciones operativas en esa semana. Por eso las ventas observadas deben leerse como un piso de la demanda real, no como una medida exacta de la falta de interés del mercado.

# 4. Supuestos explícitos del escenario

In [6]:
# Supuestos académicos del caso (no vienen de UCI): capacidad, servicio mínimo y concentración máxima
CAPACITY_RATIO = 0.70
MIN_SERVICE_RATIO = 0.20
MAX_CONCENTRATION_RATIO = 0.35
# Se derivan los límites en unidades enteras a partir de la demanda estimada
demanda_total = int(parametros["demanda"].sum())
capacidad_total = int(np.floor(CAPACITY_RATIO * demanda_total))
limite_concentracion = int(np.floor(MAX_CONCENTRATION_RATIO * capacidad_total))
# Cada SKU debe recibir como mínimo el 20 % de su demanda (y al menos una unidad)
parametros["min_servicio"] = np.maximum(
    np.ceil(MIN_SERVICE_RATIO * parametros["demanda"]).astype(int), 1
)
# Se comprueba que el problema tiene solución: los mínimos caben en la capacidad y en el tope por SKU
assert parametros["min_servicio"].sum() <= capacidad_total
assert (parametros["min_servicio"] <= limite_concentracion).all()
# Se muestran la demanda y la capacidad resultantes, y la tabla de parámetros
print("Demanda:", demanda_total, "· capacidad:", capacidad_total)
display(parametros)

Demanda: 9982 · capacidad: 6987


,demanda,price,weeks_present,demand_total,min_servicio
StockCode,,,,,
22197,1809,0.85,12,21703,362
84077,1223,0.29,12,14676,245
85099B,1175,2.08,12,14098,235
22086,1080,2.95,12,12951,216
23084,959,2.08,12,11505,192
84879,860,1.69,12,10310,172
22952,765,0.55,12,9177,153
22577,746,0.29,12,8948,150
22578,693,0.29,12,8305,139


**Interpretación de resultados**

La salida fija los parámetros de partida del modelo: una demanda total esperada de 9.982 unidades y una capacidad de 6.987 unidades, que equivale al 70 % supuesto en el caso. La tabla muestra, para cada uno de los 10 SKU, la demanda semanal promedio (redondeada hacia arriba), el precio mediano (resistente a atípicos y promociones) y el servicio mínimo obligatorio (20 % de la demanda semanal). Además verifica que el problema tiene solución: los mínimos suman 1.899 unidades, muy por debajo de la capacidad, así que hay margen para que el solucionador reparta el resto del inventario cumpliendo las restricciones del negocio.

> **Importante:** 70 % de capacidad, 20 % de servicio mínimo y 35 % de concentración son supuestos académicos del caso. No provienen de UCI.

### Pregunta 3
Separe parámetros derivados de UCI y supuestos de operaciones.

**Respuesta:**

Provienen de UCI la demanda semanal promedio (redondeada hacia arriba), el precio mediano por SKU, las semanas de presencia del producto y la demanda total acumulada en las 12 semanas, todos calculados a partir de las transacciones históricas ya limpias. Son supuestos de operaciones, definidos de forma explícita como reglas académicas y no extraídos de los datos, la capacidad total (70 % de la demanda esperada), el servicio mínimo por SKU (20 % de su demanda) y el tope de concentración por producto (35 % de la capacidad total).

# 5. Línea base intuitiva: asignación proporcional a la demanda

In [7]:
# Línea base intuitiva: reparte la capacidad en proporción a la demanda, respetando mínimos, demanda y tope por SKU
def proportional_allocation(params, capacity, min_col="min_servicio", max_conc=None):
    """Regla proporcional con mínimos, demanda y concentración: baseline factible."""
    # Tope de cada SKU: su demanda y, si existe, el límite de concentración
    upper = params["demanda"].astype(int).copy()
    if max_conc is not None:
        upper = upper.clip(upper=int(max_conc))
    # Piso de cada SKU: el servicio mínimo
    lower = params[min_col].astype(int).copy()
    # Si las reglas no permiten asignar toda la capacidad, se avisa en lugar de comparar algo injusto
    if (lower > upper).any() or lower.sum() > capacity or upper.sum() < capacity:
        raise ValueError("La política no permite asignar toda la capacidad.")
    # Se parte de los mínimos y se reparte el resto en proporción a la demanda
    alloc = lower.copy()
    remaining = int(capacity - alloc.sum())
    weights = params["demanda"].astype(float)
    # Se repite hasta repartir todas las unidades, respetando el espacio que le queda a cada SKU
    while remaining:
        room = upper - alloc
        active = room > 0
        if not active.any():
            break
        quota = remaining * weights[active] / weights[active].sum()
        add = np.minimum(np.floor(quota).astype(int), room[active])
        if add.sum() == 0:
            order = (quota - np.floor(quota)).sort_values(ascending=False).index
            for sku in order:
                if remaining == 0: break
                if room.loc[sku] > 0:
                    alloc.loc[sku] += 1
                    remaining -= 1
        else:
            alloc.loc[active] += add
            remaining -= int(add.sum())
    # Verificación final: se asignó exactamente la capacidad
    assert alloc.sum() == capacity
    return alloc

# Se aplica la regla y se calcula su valor potencial: es la referencia que PuLP debe superar
baseline = parametros.copy()
baseline["asignacion_intuitiva"] = proportional_allocation(
    parametros, capacidad_total, max_conc=limite_concentracion
)
baseline["valor_intuitivo"] = baseline["asignacion_intuitiva"] * baseline["price"]
display(baseline)
print("Unidades intuitivas:", int(baseline["asignacion_intuitiva"].sum()))
print("Valor potencial intuitivo (£):", round(baseline["valor_intuitivo"].sum(), 2))

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_intuitiva,valor_intuitivo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1266,1076.10
84077,1223,0.29,12,14676,245,857,248.53
85099B,1175,2.08,12,14098,235,823,1711.84
22086,1080,2.95,12,12951,216,756,2230.20
23084,959,2.08,12,11505,192,672,1397.76
84879,860,1.69,12,10310,172,601,1015.69
22952,765,0.55,12,9177,153,535,294.25
22577,746,0.29,12,8948,150,522,151.38
22578,693,0.29,12,8305,139,485,140.65


Unidades intuitivas: 6987
Valor potencial intuitivo (£): 8853.9


**Interpretación de resultados**

Se presenta la línea base intuitiva: las 6.987 unidades de capacidad (70 % de la demanda total) se reparten entre los 10 SKU en proporción a su demanda, cumpliendo el servicio mínimo (20 %) y el tope de concentración (35 %). La tabla indica la asignación y el valor potencial de cada producto, con un total de £8.853,90. Esa cifra es el punto de comparación contra el cual se medirá el modelo de PuLP, para saber cuánto aporta realmente la optimización frente a una regla de reparto convencional.

### Pregunta 4
¿Por qué la comparación con PuLP sería engañosa si la regla intuitiva excediera un límite?

**Respuesta:**

La comparación sería engañosa porque PuLP solo evalúa soluciones factibles, es decir, que cumplan a la vez capacidad total, demanda por SKU, servicio mínimo y concentración máxima. Si la regla intuitiva superara alguno de esos límites, propondría una asignación imposible de ejecutar en la operación real y la comparación perdería validez. El código evita esto al comprobar que la asignación proporcional respete los mismos topes que el modelo; así, la diferencia de valor refleja solo la calidad de la decisión y no el incumplimiento de restricciones operativas.

# 6. Formulación con PuLP

In [8]:
# Modelo: maximizar el valor potencial de venta (precio × unidades asignadas)
model = pl.LpProblem("Asignacion_Inventario", pl.LpMaximize)
# Variable de decisión: unidades enteras (mayores o iguales a 0) que se asignan a cada SKU
x = {sku: pl.LpVariable(f"x_{i}", lowBound=0, cat=pl.LpInteger)
     for i, sku in enumerate(parametros.index)}
# Función objetivo
model += pl.lpSum(parametros.loc[sku, "price"] * x[sku] for sku in parametros.index)
# Restricción de capacidad: no se puede asignar más de lo que hay
model += pl.lpSum(x.values()) <= capacidad_total
# Por cada SKU: no más que su demanda, no menos que su servicio mínimo y no más que el tope de concentración
for sku in parametros.index:
    model += x[sku] <= int(parametros.loc[sku, "demanda"])
    model += x[sku] >= int(parametros.loc[sku, "min_servicio"])
    model += x[sku] <= limite_concentracion
# Con 10 SKU se esperan 10 variables y 31 restricciones (1 de capacidad y 3 por SKU)
print("Variables:", len(x), "· restricciones:", len(model.constraints))

Variables: 10 · restricciones: 31


**Interpretación de resultados**

La salida confirma que el modelo quedó bien planteado en PuLP. Las 10 variables de decisión son las unidades enteras asignadas a cada SKU. Las 31 restricciones reproducen las reglas del caso: una de capacidad total y tres por cada SKU (tope de demanda, servicio mínimo y límite de concentración). De este modo, el solucionador solo explorará soluciones que respeten todas las condiciones del problema.

### Pregunta 5
Explique por qué maximizar precio × unidades sin restricciones no responde la decisión de negocio.

**Respuesta:**

Maximizar precio × unidades sin restricciones no responde a la decisión de negocio porque el modelo intentaría asignar unidades sin límite al producto de mayor precio, algo físicamente imposible. En la práctica existen limitaciones reales que deben expresarse como restricciones para que la solución sea ejecutable: la capacidad total de inventario, la demanda máxima esperada por SKU, el servicio mínimo para todos los productos y los topes de concentración que evitan depender de un único artículo. Sin esas reglas, la solución no sería operativamente factible ni representaría las condiciones reales de la empresa.

# 7. Resolver el problema

In [9]:
# Se elige el solucionador CBC, que viene con PuLP, y se resuelve el modelo
solver = pl.PULP_CBC_CMD(msg=False)
status = model.solve(solver)

# El estado dice si hay solución (Optimal, Infeasible, Unbounded o Not Solved); se lee antes que los números
print("Estado:", pl.LpStatus[status])
print("Valor objetivo (£):", round(pl.value(model.objective), 2))

Estado: Optimal
Valor objetivo (£): 11478.03


**Interpretación de resultados**

El solucionador CBC terminó con estado "Optimal", lo que significa que halló una solución factible que maximiza el valor potencial de venta bajo las restricciones definidas. El valor objetivo es £11,478.03: el mayor valor bruto posible al repartir las 6,987 unidades entre los 10 SKU cumpliendo a la vez capacidad, demanda máxima por producto, servicio mínimo de 20 % y concentración de 35 %. Es la solución óptima del modelo entero y la referencia para medir la mejora sobre la regla proporcional.

In [10]:
# No se sigue si el solucionador no encontró un óptimo
if pl.LpStatus[status] != "Optimal":
    raise RuntimeError(
        "El modelo no alcanzó estado Optimal. Revise capacidad, mínimos y concentración."
    )

# Se guarda la asignación óptima por SKU y su valor potencial
opt = parametros.copy()
opt["asignacion_optima"] = [int(round(x[sku].value())) for sku in opt.index]
opt["valor_optimo"] = opt["asignacion_optima"] * opt["price"]

# Se muestran la tabla y los totales: las unidades asignadas deben igualar la capacidad
display(opt)
print("Unidades asignadas:", int(opt["asignacion_optima"].sum()))
print("Valor potencial óptimo (£):", round(opt["valor_optimo"].sum(), 2))

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_optima,valor_optimo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1554,1320.90
84077,1223,0.29,12,14676,245,245,71.05
85099B,1175,2.08,12,14098,235,1175,2444.00
22086,1080,2.95,12,12951,216,1080,3186.00
23084,959,2.08,12,11505,192,959,1994.72
84879,860,1.69,12,10310,172,860,1453.40
22952,765,0.55,12,9177,153,153,84.15
22577,746,0.29,12,8948,150,150,43.50
22578,693,0.29,12,8305,139,139,40.31


Unidades asignadas: 6987
Valor potencial óptimo (£): 11478.03


**Interpretación de resultados**

La tabla muestra la asignación óptima de PuLP para las 6.987 unidades, que produce un valor potencial de £11,478.03. Se aprecia la lógica económica del modelo: los productos más caros (22086 a £2.95; 85099B y 23084 a £2.08) reciben toda su demanda estimada, mientras que los más baratos (84077, 22577 y 22578 a £0.29) quedan en el piso de servicio mínimo, el 20 %. Es decir, con inventario escaso el solucionador privilegia el valor por unidad, sin dejar de respetar capacidad, demanda máxima ni servicio mínimo.

### Pregunta 6
¿Qué garantiza el estado Optimal y qué no garantiza respecto de la demanda y los costos?

**Respuesta:**

El estado "Optimal" asegura que, dentro de las restricciones declaradas (capacidad total, demanda máxima por SKU, servicio mínimo y concentración máxima), no existe otra asignación factible con mayor valor potencial de venta. No asegura, en cambio, que los parámetros de entrada, como la demanda estimada con ventas históricas, reflejen la demanda real del mercado, ni que el resultado se convierta en utilidad: la función objetivo solo maximiza el valor bruto de lo asignado y el modelo no incluye costos operativos, de adquisición ni de almacenamiento.

# 8. Comparación: intuitivo vs óptimo

In [11]:
# Se juntan en una tabla la asignación intuitiva y la óptima, por SKU
comparison = parametros[["demanda","price"]].copy()
comparison["intuitivo"] = baseline["asignacion_intuitiva"]
comparison["optimo"] = opt["asignacion_optima"]
# Diferencia de unidades: positivo significa que el modelo asigna más que la regla intuitiva
comparison["dif_unidades"] = comparison["optimo"] - comparison["intuitivo"]
comparison["valor_intuitivo"] = comparison["intuitivo"] * comparison["price"]
comparison["valor_optimo"] = comparison["optimo"] * comparison["price"]

# Se ordena de mayor a menor cambio para ver quién gana y quién pierde unidades
display(comparison.sort_values("dif_unidades", ascending=False))

,demanda,price,intuitivo,optimo,dif_unidades,valor_intuitivo,valor_optimo
StockCode,,,,,,,
85099B,1175,2.08,823,1175,352,1711.84,2444.00
22086,1080,2.95,756,1080,324,2230.20,3186.00
22197,1809,0.85,1266,1554,288,1076.10,1320.90
23084,959,2.08,672,959,287,1397.76,1994.72
84879,860,1.69,601,860,259,1015.69,1453.40
21915,672,1.25,470,672,202,587.50,840.00
22578,693,0.29,485,139,-346,140.65,40.31
22577,746,0.29,522,150,-372,151.38,43.50
22952,765,0.55,535,153,-382,294.25,84.15


**Interpretación de resultados**

La tabla compara ambas asignaciones y revela cómo PuLP redistribuye las 6.987 unidades. Aumenta lo asignado a los SKU de mayor precio (22086 a £2.95 y 85099B a £2.08) hasta cubrir su demanda estimada y recorta con fuerza los de menor precio (84077, 22577 y 22578 a £0.29), que bajan justo al mínimo de servicio de 20 %. Con ello busca el máximo valor potencial (£11,478.03) sin romper servicio mínimo ni concentración, a diferencia de la regla intuitiva, que reparte proporcionalmente a la demanda sin considerar cuánto vale cada producto.

In [12]:
# Valor total de cada solución
v_int = float(comparison["valor_intuitivo"].sum())
v_opt = float(comparison["valor_optimo"].sum())
# Mejora absoluta (en £) y relativa (frente a la regla intuitiva)
improvement = v_opt - v_int
improvement_pct = improvement / v_int if v_int else np.nan

# Se informa siempre en £ y en porcentaje: es valor potencial de venta, no utilidad
print("Valor intuitivo (£):", round(v_int, 2))
print("Valor óptimo (£):", round(v_opt, 2))
print("Mejora absoluta (£):", round(improvement, 2))
print("Mejora porcentual:", f"{improvement_pct:.2%}")

Valor intuitivo (£): 8853.9
Valor óptimo (£): 11478.03
Mejora absoluta (£): 2624.13
Mejora porcentual: 29.64%


**Interpretación de resultados**

Aquí se mide cuánto gana el modelo frente a la regla intuitiva. PuLP alcanza £11,478.03 de valor potencial, es decir £2,624.13 más que los £8,853.90 de la distribución proporcional, una mejora relativa de 29.64 %. La diferencia surge de mover el inventario escaso hacia los productos de mayor precio unitario sin salirse del servicio mínimo ni de la concentración, aprovechando mejor las 6,987 unidades disponibles. Conviene recordar que se trata de valor potencial bruto, no de utilidad neta ni de ingreso asegurado: que se concrete depende de que la demanda real se parezca a la estimada con datos históricos.

In [13]:
# Se reordena la tabla en formato largo para graficar los dos métodos lado a lado
long = (
    comparison[["intuitivo","optimo"]]
    .reset_index()
    .melt(id_vars="StockCode", var_name="metodo", value_name="unidades")
)

# Barras agrupadas: unidades asignadas por SKU, regla intuitiva frente a óptimo
fig = px.bar(
    long,
    x="StockCode",
    y="unidades",
    color="metodo",
    barmode="group",
    title="Asignación intuitiva vs óptima"
)
fig.show()

**Interpretación de resultados**

El gráfico de barras agrupadas permite ver cómo se reparten las 6.987 unidades entre los 10 SKU. En los productos más caros (22086 a £2.95; 85099B y 23084 a £2.08) la barra roja (óptimo) supera a la azul (intuitivo), pues PuLP les asigna toda su demanda estimada. En los de menor valor (84077, 22952, 22577 y 22578, a £0.29 o £0.55) la roja cae al piso de servicio mínimo (20 %), mientras que la azul los infla por seguir la proporción de la demanda. El SKU 22197 tiene la mayor demanda, pero su precio bajo (£0.85) lo deja en una posición intermedia. En resumen, la optimización no persigue igualdad de cantidades, sino concentrar el inventario limitado donde genera más valor.

### Pregunta 7
¿Qué SKU ganan y pierden unidades, y cuál es la lógica económica de ese cambio?

**Respuesta:**

Ganan unidades los SKU de precio unitario más alto, como 22086 (£2.95), 85099B (£2.08) y 23084 (£2.08), y pierden unidades los de menor valor, como 84077, 22577 y 22578 (£0.29). La lógica económica es que el modelo redistribuye la capacidad limitada (6,987 unidades) para maximizar el valor potencial de venta: recorta los productos baratos hasta su servicio mínimo (20 % de su demanda) y libera inventario para los más rentables, elevando el valor total sin violar los límites de concentración ni de demanda máxima.

# 9. Verificación de restricciones

In [14]:
# Holgura = límite − uso: cuánto sobra en cada regla; 0 significa restricción activa
checks = pd.DataFrame(index=parametros.index)
checks["x"] = opt["asignacion_optima"]
checks["demanda"] = parametros["demanda"]
checks["min_servicio"] = parametros["min_servicio"]
checks["holgura_demanda"] = checks["demanda"] - checks["x"]
checks["holgura_minimo"] = checks["x"] - checks["min_servicio"]
checks["holgura_concentracion"] = limite_concentracion - checks["x"]
display(checks)
# Verificaciones: no pasar la capacidad y ninguna holgura negativa (sería una regla violada)
assert checks["x"].sum() <= capacidad_total
assert (checks[["holgura_demanda","holgura_minimo","holgura_concentracion"]] >= 0).all().all()
# Resumen: capacidad usada y si la concentración es vinculante (holgura 0 en algún SKU)
print("Capacidad usada:", int(checks["x"].sum()), "/", capacidad_total)
print("Límite de concentración vinculante:", bool((checks["holgura_concentracion"] == 0).any()))

,x,demanda,min_servicio,holgura_demanda,holgura_minimo,holgura_concentracion
StockCode,,,,,,
22197,1554,1809,362,255,1192,891
84077,245,1223,245,978,0,2200
85099B,1175,1175,235,0,940,1270
22086,1080,1080,216,0,864,1365
23084,959,959,192,0,767,1486
84879,860,860,172,0,688,1585
22952,153,765,153,612,0,2292
22577,150,746,150,596,0,2295
22578,139,693,139,554,0,2306


Capacidad usada: 6987 / 6987
Límite de concentración vinculante: False


**Interpretación de resultados**

La tabla de verificación demuestra que la solución óptima cumple todas las reglas del caso. La holgura de demanda es cero en los SKU más caros (85099B, 22086 y 23084): se cubre toda su demanda estimada. La holgura de servicio mínimo es cero en los más baratos (84077, 22952, 22577 y 22578): quedan exactamente en el 20 % obligatorio, lo que libera inventario para los productos más rentables. La holgura de concentración es positiva en todos los SKU (por ejemplo, 891 unidades en el 22197), de modo que el tope de 35 % no es vinculante y no condicionó la solución. El resumen confirma además el uso total de la capacidad, 6.987 de 6.987 unidades, sin romper ninguna regla operativa.

### Pregunta 8
¿La concentración del 35 % es vinculante en el escenario base? Sustente con las holguras.

**Respuesta:**

No, el tope de concentración de 35 % no es vinculante en el escenario base. Las holguras de concentración son estrictamente positivas en todos los SKU. El producto con mayor asignación óptima (SKU 22197) recibe 1 554 unidades y queda 891 unidades por debajo del máximo permitido de 2 445. Como ningún SKU llega al tope, esta regla no condiciona la solución óptima encontrada.

# 10. Sensibilidad a la capacidad

In [15]:
# Resuelve el mismo modelo con otros supuestos: devuelve el estado, el valor óptimo, el valor de la línea base y la asignación
def solve_scenario(params, capacity_ratio=0.70, min_service_ratio=0.20,
                   max_conc_ratio=0.35):
    # Se recalculan la capacidad y el tope de concentración del escenario
    total = int(params["demanda"].sum())
    capacity = int(np.floor(capacity_ratio * total))
    max_conc = int(np.floor(max_conc_ratio * capacity))
    p = params.copy()
    # Servicio mínimo del escenario (al menos una unidad por SKU)
    p["min_scn"] = np.maximum(np.ceil(min_service_ratio * p["demanda"]).astype(int), 1)
    # Si los mínimos no caben, el escenario es infactible y no se resuelve
    if p["min_scn"].sum() > capacity or (p["min_scn"] > max_conc).any():
        return {"status":"Infeasible", "capacity":capacity, "optimal_value":np.nan,
                "baseline_value":np.nan, "uplift_pct":np.nan, "allocation":None}
    # Mismo modelo entero del laboratorio, con los límites del escenario
    m = pl.LpProblem("Escenario", pl.LpMaximize)
    z = {sku: pl.LpVariable(f"z_{i}", lowBound=0, cat=pl.LpInteger)
         for i,sku in enumerate(p.index)}
    m += pl.lpSum(p.loc[sku,"price"]*z[sku] for sku in p.index)
    m += pl.lpSum(z.values()) <= capacity
    for sku in p.index:
        m += z[sku] <= int(p.loc[sku,"demanda"])
        m += z[sku] >= int(p.loc[sku,"min_scn"])
        m += z[sku] <= max_conc
    # Se resuelve y se lee el estado antes de usar los valores
    status = pl.LpStatus[m.solve(pl.PULP_CBC_CMD(msg=False))]
    if status != "Optimal":
        return {"status":status,"capacity":capacity,"optimal_value":np.nan,
                "baseline_value":np.nan,"uplift_pct":np.nan,"allocation":None}
    # Asignación óptima y línea base proporcional bajo las mismas reglas, para comparar con justicia
    allocation = pd.Series({sku:int(round(z[sku].value())) for sku in p.index})
    baseline_scn = proportional_allocation(p, capacity, min_col="min_scn", max_conc=max_conc)
    optimal_value = float((allocation*p["price"]).sum())
    baseline_value = float((baseline_scn*p["price"]).sum())
    return {"status":status,"capacity":capacity,"optimal_value":optimal_value,
            "baseline_value":baseline_value,
            "uplift_pct":100*(optimal_value/baseline_value-1),
            "allocation":allocation.to_dict()}

# Se resuelven los tres escenarios de capacidad: 60 %, 70 % y 85 % de la demanda
scenarios = [solve_scenario(parametros, capacity_ratio=r) for r in [0.60,0.70,0.85]]
# Se arma la tabla de resultados por escenario
scenario_df = pd.DataFrame([
    {"capacity_ratio": ratio, **{k:v for k,v in result.items() if k != "allocation"}}
    for ratio, result in zip([0.60,0.70,0.85], scenarios)
])
# solve_scenario devuelve uplift_pct como 40.07; Plotly necesita 0.4007 para el formato %.
scenario_df["improvement_pct"] = scenario_df["uplift_pct"] / 100
display(scenario_df)

,capacity_ratio,status,capacity,optimal_value,baseline_value,uplift_pct,improvement_pct
0,0.60,Optimal,5989,10629.73,7588.60,40.074981,0.400750
1,0.70,Optimal,6987,11478.03,8853.90,29.638126,0.296381
2,0.85,Optimal,8484,12214.08,10751.27,13.605927,0.136059


**Interpretación de resultados**

La tabla analiza la sensibilidad a la capacidad con escenarios de 60 %, 70 % y 85 % de la demanda total. Al crecer la capacidad (de 5.989 a 8.484 unidades) suben tanto el valor de la regla intuitiva como el óptimo de PuLP, algo esperable al haber más recursos. En cambio, la mejora porcentual de PuLP se reduce con fuerza: 40,07 % en el escenario más escaso (60 %) frente a 13,61 % en el más holgado (85 %). Es decir, optimizar aporta más cuando los recursos son muy limitados, porque dirigir las unidades a los SKU más caros hace una diferencia grande. A medida que la capacidad se acerca a la demanda total, la regla proporcional cubre más volumen por sí sola y la brecha relativa se cierra. El estado "Optimal" en los tres casos indica que el modelo es factible y estable ante distintas condiciones operativas.

In [16]:
# Gráfico de la ventaja de optimizar según la capacidad disponible (60 %, 70 % y 85 %)
fig = px.bar(
    scenario_df,
    x="capacity_ratio",
    y="improvement_pct",
    text=scenario_df["improvement_pct"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "NA"),
    title="Ventaja de optimizar según capacidad disponible",
    labels={"capacity_ratio":"Capacidad / demanda", "improvement_pct":"Mejora vs intuitivo"}
)
fig.show()

**Interpretación de resultados**

El gráfico muestra cómo decrece la ventaja relativa de PuLP sobre la regla proporcional según la capacidad. Cuanto menos inventario hay, más rinde optimizar: con 60 % de la demanda cubierta, PuLP supera a la regla intuitiva en 40,1 %; con 70 % la brecha es de 29,6 % y con 85 % baja a 13,6 %. El mayor valor de la programación entera aparece entonces en contextos de fuerte escasez, donde destinar unidades a los SKU de mayor precio pesa mucho. Cuando la capacidad se aproxima a la demanda total, la regla intuitiva atiende más volumen y la diferencia relativa se achica, aunque el valor potencial absoluto siga subiendo en ambos métodos.

### Pregunta 9
¿Por qué la mejora porcentual de PuLP cambia cuando la capacidad pasa de 60 % a 85 %?

**Respuesta:**

La mejora de PuLP baja de 40,07 % (60 %) a 29,64 % (70 %) y a 13,61 % (85 %) porque, a mayor capacidad, la regla proporcional se acerca cada vez más al óptimo. Con poca capacidad (60 %), PuLP saca partido de la escasez concentrando las unidades en los SKU más caros, mientras que la regla intuitiva gasta inventario en productos baratos por repartir según la demanda. Al acercarse la capacidad a 85 %, la escasez disminuye, ambos métodos cubren más demanda y la brecha de valor se estrecha; por eso cae la ventaja relativa de PuLP aunque su valor absoluto siga aumentando.

# 11. Decisión y recomendación

### Pregunta 10
¿Recomendaría operar con la asignación PuLP? Indique una condición que podría cambiar la respuesta.

**Respuesta:**

Sí, se recomienda operar con la asignación de PuLP: eleva el valor potencial de venta en 29,64 % respecto a la regla intuitiva y cumple estrictamente capacidad, servicio mínimo y concentración. Aun así, la recomendación podría cambiar si el objetivo de la empresa fuera maximizar utilidad o rentabilidad y no el valor bruto de ventas. Como el modelo solo usa el precio de venta y no incluye costos de adquisición ni márgenes de contribución por SKU, la mejor asignación en términos de rentabilidad podría ser distinta si los productos más caros tuvieran márgenes menores o costos de almacenamiento más altos.

# 12. Retos individuales de aplicación

La parte guiada ya formuló y verificó el modelo. Resuelva los tres retos en su notebook personal. Son **10 preguntas** en total (4 + 3 + 3). En cada respuesta distinga resultados calculados, supuestos y recomendación. No altere las transacciones para producir una mejora mayor.

## Reto 1 — ¿Conviene ampliar la capacidad? (4 preguntas)

**Caso.** Operaciones puede aumentar la capacidad semanal, pero debe pagar por ella. Compare 60 %, 70 % y 85 % de la demanda total. Mantenga las demás reglas.

**R1.1.** Construya una tabla con capacidad, valor intuitivo, valor PuLP y mejora porcentual para los tres escenarios. ¿Qué representa cada cifra?

La tabla resume cómo rinden dos estrategias de asignación en tres niveles de escasez. "Capacidad" son las unidades disponibles en total (60 %, 70 % y 85 % de la demanda esperada). "Valor Intuitivo" es el potencial de venta que resulta de repartir el inventario proporcionalmente a la demanda cumpliendo las restricciones. "Valor PuLP" es el máximo potencial alcanzable con optimización matemática. "Mejora (%)" expresa la ventaja relativa del modelo prescriptivo sobre la regla tradicional en cada nivel de capacidad.


**R1.2.** Calcule el valor marginal por unidad adicional de capacidad entre 60 %→70 % y 70 %→85 %. ¿Por qué no son iguales?

El valor marginal de la capacidad baja de £0,85 por unidad (de 60 % a 70 %) a £0,49 por unidad (de 70 % a 85 %). La diferencia se explica por rendimientos decrecientes, propios de un modelo entero con topes de demanda: con alta escasez (60 %), cada unidad extra va a los SKU de mayor precio y aporta mucho valor. Cuando la capacidad se holga (85 %), la demanda de los productos más rentables ya está cubierta, así que las unidades adicionales van a SKU de menor valor o chocan con el tope de demanda, y cada una aporta menos.


**R1.3.** Si una unidad adicional de capacidad cuesta £0,65 por semana, ¿cuál ampliación conviene respecto del escenario de 60 %? Muestre valor incremental, costo incremental y saldo neto.

Frente al escenario base de 60 %, conviene ampliar hasta 70 %. Esto implica 998 unidades adicionales a £0,65, con un costo incremental de £648,70, mientras PuLP aumenta el valor potencial en £848,30, para un saldo neto positivo de £199,60. En cambio, llegar a 85 % no resulta rentable: las 2.495 unidades extra cuestan £1.621,75 y el valor adicional es de £1.584,35, lo que deja un saldo neto negativo de £-37,40.


**R1.4.** Recomiende a operaciones una capacidad e indique una limitación de su cálculo.

Se recomienda a operaciones fijar la capacidad en 70 % (6.987 unidades), porque es la que maximiza el saldo neto entre el costo de adquirir capacidad y el valor potencial optimizado. Una limitación clave es que la función objetivo maximiza valor bruto (precio × unidades) y no utilidad neta; por eso no se consideran otros costos de la ampliación, como mantener inventario, ni la posibilidad de que la demanda real difiera de la estimación histórica usada como parámetro.

In [17]:
# =======================================
# RETO 1: ¿Conviene ampliar la capacidad?
# =======================================

# R1.1: Tabla de escenarios (reutilizando scenario_df del laboratorio)
tabla_r1_1 = scenario_df[['capacity_ratio', 'capacity', 'baseline_value', 'optimal_value', 'uplift_pct']].copy()
tabla_r1_1.columns = ['Ratio Capacidad', 'Capacidad (unidades)', 'Valor Intuitivo (£)', 'Valor PuLP (£)', 'Mejora (%)']
tabla_r1_1['Mejora (%)'] = tabla_r1_1['Mejora (%)'].map('{:.2f}%'.format)

print("=" * 38)
print("R1.1: Tabla de Escenarios de Capacidad")
print("=" * 38)

display(tabla_r1_1)
print("Nota: 'Valor Intuitivo' es la asignación proporcional a la demanda. 'Valor PuLP' es la asignación optimizada.")

# R1.2: Valor marginal por unidad adicional de capacidad (en el modelo óptimo)
cap_60, val_60 = scenario_df.loc[0, 'capacity'], scenario_df.loc[0, 'optimal_value']
cap_70, val_70 = scenario_df.loc[1, 'capacity'], scenario_df.loc[1, 'optimal_value']
cap_85, val_85 = scenario_df.loc[2, 'capacity'], scenario_df.loc[2, 'optimal_value']

vm_60_70 = (val_70 - val_60) / (cap_70 - cap_60)
vm_70_85 = (val_85 - val_70) / (cap_85 - cap_70)

print("\n")
print("=" * 50)
print("R1.2: Valor Marginal de la Capacidad (Modelo PuLP)")
print("=" * 50)
print(f"• Valor marginal (60% → 70%): £{vm_60_70:.4f} por cada unidad adicional de capacidad.")
print(f"• Valor marginal (70% → 85%): £{vm_70_85:.4f} por cada unidad adicional de capacidad.")

# R1.3: Análisis de costo incremental (£0.65 por unidad) respecto al escenario base de 60%
costo_unitario = 0.65

# Comparación 70% vs 60%
inc_cap_70 = cap_70 - cap_60
inc_costo_70 = inc_cap_70 * costo_unitario
inc_valor_70 = val_70 - val_60
saldo_neto_70 = inc_valor_70 - inc_costo_70

# Comparación 85% vs 60%
inc_cap_85 = cap_85 - cap_60
inc_costo_85 = inc_cap_85 * costo_unitario
inc_valor_85 = val_85 - val_60
saldo_neto_85 = inc_valor_85 - inc_costo_85

print("\n")
print("=" * 53)
print("R1.3: Análisis Incremental (vs escenario base de 60%)")
print("=" * 53)
print(f"Ampliación a 70%:")
print(f"  - Unidades extra: {inc_cap_70}")
print(f"  - Costo incremental: £{inc_costo_70:.2f}")
print(f"  - Valor incremental (PuLP): £{inc_valor_70:.2f}")
print(f"  - Saldo neto: £{saldo_neto_70:.2f}")
print(f"\nAmpliación a 85%:")
print(f"  - Unidades extra: {inc_cap_85}")
print(f"  - Costo incremental: £{inc_costo_85:.2f}")
print(f"  - Valor incremental (PuLP): £{inc_valor_85:.2f}")
print(f"  - Saldo neto: £{saldo_neto_85:.2f}")

R1.1: Tabla de Escenarios de Capacidad


,Ratio Capacidad,Capacidad (unidades),Valor Intuitivo (£),Valor PuLP (£),Mejora (%)
0,0.60,5989,7588.60,10629.73,40.07%
1,0.70,6987,8853.90,11478.03,29.64%
2,0.85,8484,10751.27,12214.08,13.61%


Nota: 'Valor Intuitivo' es la asignación proporcional a la demanda. 'Valor PuLP' es la asignación optimizada.


R1.2: Valor Marginal de la Capacidad (Modelo PuLP)
• Valor marginal (60% → 70%): £0.8500 por cada unidad adicional de capacidad.
• Valor marginal (70% → 85%): £0.4917 por cada unidad adicional de capacidad.


R1.3: Análisis Incremental (vs escenario base de 60%)
Ampliación a 70%:
  - Unidades extra: 998
  - Costo incremental: £648.70
  - Valor incremental (PuLP): £848.30
  - Saldo neto: £199.60

Ampliación a 85%:
  - Unidades extra: 2495
  - Costo incremental: £1621.75
  - Valor incremental (PuLP): £1584.35
  - Saldo neto: £-37.40


**Interpretación de resultados**

Los resultados reflejan la sensibilidad del modelo a la disponibilidad de inventario (60 %, 70 % y 85 % de la demanda). El valor potencial absoluto crece con la capacidad, pero la ventaja relativa de PuLP frente a la regla intuitiva se reduce notablemente. El análisis incremental también muestra rendimientos decrecientes: ampliar la capacidad conviene solo hasta cierto punto, a partir del cual el costo de la capacidad adicional supera el valor extra que el modelo logra generar.

## Reto 2 — Servicio mínimo y costo de la equidad (3 preguntas)

**Caso.** Comercial propone elevar el servicio mínimo para evitar productos casi desabastecidos. Evalúe 20 %, 30 % y 50 % con capacidad 70 %.

**R2.1.** Resuelva los tres escenarios y muestre valor óptimo y unidades por SKU. ¿Cuál sigue siendo factible?

Los tres escenarios (20 %, 30 % y 50 % de servicio mínimo) terminan en "Optimal", así que todos son factibles con capacidad de 70 %. El valor óptimo, sin embargo, va bajando a medida que se endurece la restricción: £11,478.03 (20 %), £11,306.53 (30 %) y £10,791.63 (50 %). Subir el piso obliga a reservar capacidad para los mínimos de todos los SKU, lo que resta flexibilidad para asignar unidades extra a los productos más caros y genera un costo de oportunidad visible en la caída del valor total.


**R2.2.** Calcule cuánto valor potencial se sacrifica al pasar de 20 % a 30 % y de 20 % a 50 %. Identifique qué SKU cambian más.

Pasar de 20 % a 30 % sacrifica £171.50 de valor potencial, y pasar de 20 % a 50 %, £686.40. El SKU que más cambia es el 22197: pierde 342 unidades (de 20 % a 30 %) y 649 (de 20 % a 50 %). Al exigir mínimos más altos a todos los productos, el modelo debe reasignar capacidad para cubrir los nuevos pisos de otros SKU; le quita al 22197 las unidades "extra" que tenía con 20 % y lo deja exactamente en su nuevo mínimo (905 unidades), lo que comprime la asignación global.


**R2.3.** Recomiende una política de servicio. Explique qué indicador operativo faltaría para decidirla en una empresa real.

Se recomienda mantener el servicio mínimo de 20 %, que maximiza el valor potencial de venta (£11,478.03) sin la pérdida de valor que implica subir el piso a 30 % o 50 %. Para validarlo en una empresa real faltaría un indicador de margen de contribución o costo unitario por SKU: el modelo solo maximiza valor bruto (precio × unidades), de modo que una política de servicio mayor podría justificarse si los productos baratos tuvieran márgenes porcentuales más altos, o si hubiera costos por quiebre de stock o penalidades contractuales que el modelo no recoge.

In [18]:
# =============================================
# RETO 2: Servicio mínimo y costo de la equidad
# =============================================

# R2.1: Resolver los tres escenarios de servicio mínimo (20%, 30%, 50%) con capacidad 70%
scenarios_service = [
    solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.20),
    solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.30),
    solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.50)
]

# Crear una tabla resumen con los resultados
tabla_r2_1 = pd.DataFrame([
    {
        "Servicio Mínimo": f"{int(ratio*100)}%",
        "Estado": res["status"],
        "Valor Óptimo (£)": round(res["optimal_value"], 2) if res["status"] == "Optimal" else "N/A",
        "Asignación (unidades)": res["allocation"] if res["status"] == "Optimal" else "N/A"
    }
    for ratio, res in zip([0.20, 0.30, 0.50], scenarios_service)
])


print("=" * 35)
print("R2.1: Escenarios de Servicio Mínimo")
print("=" * 35)
display(tabla_r2_1)

# R2.2: Calcular el valor potencial sacrificado y los SKU que más cambian
alloc_20 = scenarios_service[0]["allocation"]
val_20 = scenarios_service[0]["optimal_value"]

alloc_30 = scenarios_service[1]["allocation"]
val_30 = scenarios_service[1]["optimal_value"]

alloc_50 = scenarios_service[2]["allocation"]
val_50 = scenarios_service[2]["optimal_value"]

sacrificio_30 = val_20 - val_30 if val_20 and val_30 else 0
sacrificio_50 = val_20 - val_50 if val_20 and val_50 else 0

print("\n")
print("=" * 33)
print("R2.2: Valor Potencial Sacrificado")
print("=" * 33)
print(f"• Sacrificio al pasar de 20% a 30%: £{sacrificio_30:.2f}")
print(f"• Sacrificio al pasar de 20% a 50%: £{sacrificio_50:.2f}")

# Identificar qué SKU cambian más (diferencia absoluta de unidades)
if isinstance(alloc_20, dict) and isinstance(alloc_30, dict):
    cambios_30 = {sku: abs(alloc_30[sku] - alloc_20[sku]) for sku in alloc_20}
    max_cambio_30 = max(cambios_30, key=cambios_30.get)
    print(f"• SKU con mayor cambio de unidades (20% -> 30%): {max_cambio_30} (Δ {cambios_30[max_cambio_30]} unidades)")

if isinstance(alloc_20, dict) and isinstance(alloc_50, dict):
    cambios_50 = {sku: abs(alloc_50[sku] - alloc_20[sku]) for sku in alloc_20}
    max_cambio_50 = max(cambios_50, key=cambios_50.get)
    print(f"• SKU con mayor cambio de unidades (20% -> 50%): {max_cambio_50} (Δ {cambios_50[max_cambio_50]} unidades)")

R2.1: Escenarios de Servicio Mínimo


,Servicio Mínimo,Estado,Valor Óptimo (£),Asignación (unidades)
0,20%,Optimal,11478.03,"{'22197': 1554, '84077': 245, '85099B': 1175, ..."
1,30%,Optimal,11306.53,"{'22197': 1212, '84077': 367, '85099B': 1175, ..."
2,50%,Optimal,10791.63,"{'22197': 905, '84077': 612, '85099B': 1175, '..."




R2.2: Valor Potencial Sacrificado
• Sacrificio al pasar de 20% a 30%: £171.50
• Sacrificio al pasar de 20% a 50%: £686.40
• SKU con mayor cambio de unidades (20% -> 30%): 22197 (Δ 342 unidades)
• SKU con mayor cambio de unidades (20% -> 50%): 22197 (Δ 649 unidades)


**Interpretación de resultados**

Los tres escenarios de servicio mínimo (20 %, 30 % y 50 %), con capacidad de 70 %, se resuelven con éxito y todos terminan en "Optimal", por lo que son operativamente factibles. Aun así, el valor óptimo cae a medida que se endurece el piso: de £11,478.03 con 20 % a £10,791.63 con 50 %. El SKU 22197 es el que más unidades pierde (649 menos al pasar de 20 % a 50 %). Esto expresa el "costo de la equidad": garantizar una asignación mayor a productos de bajo valor consume capacidad escasa que de otro modo iría a artículos más rentables, y por eso el valor potencial total disminuye.

## Reto 3 — ¿La concentración limita de verdad? (3 preguntas)

**Caso.** Riesgos propone limitar cada SKU al 20 %, 25 %, 35 % o 50 % de la capacidad. Mantenga el escenario de capacidad 70 % y servicio mínimo 20 %.

**R3.1.** Resuelva las cuatro políticas. Presente valor óptimo, asignación y factibilidad.

Las cuatro políticas de concentración máxima (20 %, 25 %, 35 % y 50 %) terminan en "Optimal", por lo que son factibles con capacidad de 70 % y servicio mínimo de 20 %. El valor óptimo sí difiere: £11,430.93 con 20 %, y £11,478.03 con 25 %, 35 % y 50 %. Esto indica que desde 25 % el tope deja de ser vinculante y el modelo asigna igual que si no existiera, maximizando el valor potencial sin romper las reglas.


**R3.2.** Indique en qué políticas la restricción cambia la solución y calcule el valor perdido frente a 35 %. ¿Qué demuestra el escenario de 50 %?

El tope cambia la solución solo en la política de 20 %, donde el valor baja a £11,430.93, una pérdida de £47.10 respecto a 35 %. Allí el SKU 22197 queda limitado a 1,397 unidades y el modelo traslada el excedente a productos de menor valor (por ejemplo, el 22952 sube a 310 unidades) para completar la capacidad. El escenario de 50 % demuestra que relajar el límite más allá del punto de no vinculación no aporta nada: el valor sigue en £11,478.03, y el límite que el modelo usa por sí solo es de 22,2 % (las 1,554 unidades del SKU 22197).


**R3.3.** Recomiende un límite de concentración y explique por qué el porcentaje por sí solo no mide todo el riesgo de inventario.

Se recomienda un límite de 25 %: logra el mismo valor máximo (£11,478.03) que 35 % y, en teoría, diversifica mejor al evitar depender demasiado de un solo producto. Sin embargo, el porcentaje por sí solo no mide todo el riesgo de inventario, porque deja fuera la variabilidad de la demanda, el costo de obsolescencia, el margen de contribución real y los tiempos de reposición. Un 25 % en un producto de alta rotación y margen estable es un riesgo muy distinto a un 25 % en uno estacional o de baja rotación, así que gestionar el riesgo exige un análisis más amplio que la proporción de unidades.

In [19]:
# ===========================================
# RETO 3: ¿La concentración limita de verdad?
# ===========================================

# R3.1: Resolver las cuatro políticas de concentración
conc_policies = [0.20, 0.25, 0.35, 0.50]
scenarios_conc = []

for conc_ratio in conc_policies:
    result = solve_scenario(parametros, capacity_ratio=0.70,
                           min_service_ratio=0.20, max_conc_ratio=conc_ratio)
    scenarios_conc.append(result)

# Crear tabla resumen
tabla_r3_1 = pd.DataFrame([
    {
        "Concentración": f"{int(ratio*100)}%",
        "Estado": res["status"],
        "Límite por SKU": int(np.floor(ratio * res["capacity"])) if res["status"] == "Optimal" else "N/A",
        "Valor Óptimo (£)": round(res["optimal_value"], 2) if res["status"] == "Optimal" else "N/A",
        "Capacidad": res["capacity"] if res["status"] == "Optimal" else "N/A"
    }
    for ratio, res in zip(conc_policies, scenarios_conc)
])

print("=" * 33)
print("R3.1: Escenarios de Concentración")
print("=" * 33)
display(tabla_r3_1)

# R3.2: Identificar en qué políticas la restricción cambia la solución
# El escenario 35% es la referencia del laboratorio
val_35 = scenarios_conc[2]["optimal_value"]
alloc_35 = scenarios_conc[2]["allocation"]

print("\n")
print("=" * 40)
print("R3.2: Análisis de Restricción Vinculante")
print("=" * 40)
print(f"Valor de referencia (35%): £{val_35:.2f}\n")

for i, (ratio, res) in enumerate(zip(conc_policies, scenarios_conc)):
    if res["status"] == "Optimal" and res["optimal_value"] is not None:
        valor_perdido = val_35 - res["optimal_value"]
        cambia = "SÍ" if abs(valor_perdido) > 0.01 else "NO"
        print(f"Concentración {int(ratio*100)}%:")
        print(f"  - Valor óptimo: £{res['optimal_value']:.2f}")
        print(f"  - ¿Cambia vs 35%? {cambia}")
        if valor_perdido > 0.01:
            print(f"  - Valor perdido: £{valor_perdido:.2f}")

        # Verificar si algún SKU alcanza el límite de concentración
        if res["allocation"]:
            limite_sku = int(np.floor(ratio * res["capacity"]))
            skus_en_limite = [sku for sku, units in res["allocation"].items() if units >= limite_sku]
            if skus_en_limite:
                print(f"  - SKU en límite: {skus_en_limite}")
        print()

# R3.3: Mostrar asignación detallada para comparar
if scenarios_conc[0]["allocation"] and scenarios_conc[2]["allocation"]:
    print("\n")
    print("=" * 46)
    print("Comparación de asignación por SKU (20% vs 35%)")
    print("=" * 46)
    df_comp = pd.DataFrame({
        'SKU': list(parametros.index),
        'Precio': parametros['price'],
        'Demanda': parametros['demanda'].astype(int),
        'Asign_20%': [scenarios_conc[0]["allocation"].get(sku, 0) for sku in parametros.index],
        'Asign_35%': [scenarios_conc[2]["allocation"].get(sku, 0) for sku in parametros.index]
    })
    display(df_comp)

R3.1: Escenarios de Concentración


,Concentración,Estado,Límite por SKU,Valor Óptimo (£),Capacidad
0,20%,Optimal,1397,11430.93,6987
1,25%,Optimal,1746,11478.03,6987
2,35%,Optimal,2445,11478.03,6987
3,50%,Optimal,3493,11478.03,6987




R3.2: Análisis de Restricción Vinculante
Valor de referencia (35%): £11478.03

Concentración 20%:
  - Valor óptimo: £11430.93
  - ¿Cambia vs 35%? SÍ
  - Valor perdido: £47.10
  - SKU en límite: ['22197']

Concentración 25%:
  - Valor óptimo: £11478.03
  - ¿Cambia vs 35%? NO

Concentración 35%:
  - Valor óptimo: £11478.03
  - ¿Cambia vs 35%? NO

Concentración 50%:
  - Valor óptimo: £11478.03
  - ¿Cambia vs 35%? NO



Comparación de asignación por SKU (20% vs 35%)


,SKU,Precio,Demanda,Asign_20%,Asign_35%
StockCode,,,,,
22197,22197,0.85,1809,1397,1554
84077,84077,0.29,1223,245,245
85099B,85099B,2.08,1175,1175,1175
22086,22086,2.95,1080,1080,1080
23084,23084,2.08,959,959,959
84879,84879,1.69,860,860,860
22952,22952,0.55,765,310,153
22577,22577,0.29,746,150,150
22578,22578,0.29,693,139,139


**Interpretación de resultados**

Los cuatro escenarios de concentración máxima (20 %, 25 %, 35 % y 50 %), con capacidad de 70 % y servicio mínimo de 20 %, se resuelven correctamente y todos quedan en "Optimal", es decir, son factibles. La excepción en valor es el escenario de 20 %: £11,430.93, o sea £47.10 menos que con 35 %, porque obliga al SKU 22197, el de mayor demanda, a bajar a 1,397 unidades. Los escenarios de 25 %, 35 % y 50 % llegan al mismo valor máximo (£11,478.03), lo que indica que desde 25 % el tope de concentración deja de ser vinculante y ya no restringe la asignación óptima.

# 13. Verificaciones finales

- Revise que cada solución factible respete capacidad, demanda, servicio y concentración.
- Indique si la regla intuitiva usa exactamente las mismas restricciones.
- Compruebe que las diez preguntas de los retos tengan respuesta numérica o una decisión justificada.
- Declare una limitación: ventas observadas ≠ demanda total; valor potencial ≠ utilidad.

# 14. Cierre ejecutivo y ticket de salida

Redacte máximo 12 líneas: decisión recomendada, capacidad, valor intuitivo y óptimo, mejora, costo de ampliar capacidad, política de servicio, concentración, principal riesgo y condición que cambiaría la recomendación.

Se recomienda operar con 6,987 unidades de capacidad (70 % de la demanda), un servicio mínimo de 20 % y un tope de concentración entre 25 % y 35 %. En ese escenario PuLP genera £11,478.03 de valor potencial, 29,64 % (£2,624.13) más que la asignación intuitiva (£8,853.90). Ampliar la capacidad hasta 70 % es rentable con un costo de £0,65 por unidad, mientras que llegar a 85 % da rendimientos decrecientes y un saldo neto negativo. El principal riesgo es que el modelo maximiza el valor bruto de las ventas e ignora márgenes de contribución, costos operativos y la incertidumbre de la demanda censurada en los datos históricos.


**Ticket:** «Con 6,987 unidades de capacidad, PuLP logra £11,478.03 frente a £8,853.90 de la regla intuitiva. Recomiendo esta asignación porque maximiza el valor potencial de venta cumpliendo las restricciones operativas, con saldo neto positivo. La conclusión cambiaría si los productos de mayor precio tuvieran márgenes de contribución menores o costos de almacenamiento notablemente más altos».

# 15. Entrega y referencias

Suba el notebook ejecutado a `semana11/` de su repositorio personal y comparta el enlace de Colab. Incluya los tres retos y las diez respuestas individuales. Declare el uso de IA.

Chen, D. (2012). *Online Retail II* [dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.
Sharda, R., Delen, D. y Turban, E. *Business Intelligence, Analytics, Data Science and AI*.
Winston, W. L. *Operations Research: Applications and Algorithms*.